In [14]:
import os
os.environ["DOCTR_FRAMEWORK"] = "pytorch"

import pytesseract
import easyocr
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from PIL import Image


GROUND_TRUTH = "urine examination test result unit physical examination volume random sample colour amber yellow aspect turbid ph 5 specific gravity 1030.0 deposit + chemical examination albumin nil sugar nil acetone in urine nil bilirubin nil bile salts nil urobilinogen normal trace other finding mucus (++) blood (trace) microscopic examination rbcs 6 - 8 / hpf pus cells 8 - 10 / hpf epithelial cells ++ / hpf casts nil crystals nil amorphous material nil bilharzia ova nil other findings nil"
IMAGE_PATH = r"D:\lenovo\EMAN\1.1.1 Graduation\Graduation-Project\Data Analysis & AI\OCR\tests_analysis\Photos\urinalysis.jpeg"

def clean_and_slice_text(text):
    text = text.lower().strip()
    
    start_index = text.find("unit")
    if start_index != -1:
        text = text[start_index + 4:]
        
    # 2. تحديد نقطة النهاية (عند بداية عبارة end of report)
    # end_index = text.find("end of report")
    # if end_index != -1:
    #     text = text[:end_index]
        
    return text.strip()

def calculate_accuracy(pred_text, ground_truth):
    sliced_pred = clean_and_slice_text(pred_text)
    sliced_ground = clean_and_slice_text(ground_truth)
    
    pred_words = set(sliced_pred.split())
    ground_words = set(sliced_ground.split())
    
    if not ground_words: return 0.0
    
    intersection = pred_words.intersection(ground_words)
    return (len(intersection) / len(ground_words)) * 100

# ----------------- [ 1. Tesseract Engine ] -----------------
def run_tesseract_with_conf(img_path):
    try:
        data = pytesseract.image_to_data(Image.open(img_path), lang='eng', output_type=pytesseract.Output.DICT)
        words = []
        confidences = []
        
        for i in range(len(data['text'])):
            if data['text'][i].strip() != "" and int(data['conf'][i]) != -1:
                words.append(data['text'][i])
                confidences.append(float(data['conf'][i]) / 100.0)
                
        full_text = " ".join(words)
        avg_conf = (sum(confidences) / len(confidences)) if confidences else 0.0
        return full_text, avg_conf
    except Exception as e:
        print(f"Tesseract Error: {e}")
        return "", 0.0

# ----------------- [ 2. EasyOCR Engine ] -----------------
def run_easyocr_with_conf(img_path):
    try:
        reader = easyocr.Reader(['en'], gpu=False)
        result = reader.readtext(img_path, detail=1, paragraph=False)
        
        words = []
        confidences = []
        
        for bbox, text, confidence in result:
            if text.strip() != "":
                words.append(text)
                if confidence is not None:
                    confidences.append(float(confidence))
                
        full_text = " ".join(words)
        avg_conf = (sum(confidences) / len(confidences)) if confidences else 0.0
        return full_text, avg_conf
    except Exception as e:
        print(f"EasyOCR Error: {e}")
        return "", 0.0

# ----------------- [ 3. docTR Engine ] -----------------
def run_doctr_with_conf(img_path):
    try:
        model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
        doc = DocumentFile.from_images(img_path)
        result = model(doc)
        export = result.export()
        
        full_text = []
        all_confidences = [] 
        
        for page in export['pages']:
            for block in page['blocks']:
                for line in block['lines']:
                    words_list = line['words']
                    line_text = " ".join([word['value'] for word in words_list])
                    full_text.append(line_text)
                    for word in words_list:
                        all_confidences.append(word['confidence'])
                        
        combined_text = " ".join(full_text)
        avg_conf = (sum(all_confidences) / len(all_confidences)) if all_confidences else 0.0
        return combined_text, avg_conf
    except Exception as e:
        print(f"docTR Error: {e}")
        return "", 0.0

if __name__ == "__main__":
    
    tess_text, tess_conf = run_tesseract_with_conf(IMAGE_PATH)
    easy_text, easy_conf = run_easyocr_with_conf(IMAGE_PATH)
    doctr_text, doctr_conf = run_doctr_with_conf(IMAGE_PATH)
    
    tess_acc = calculate_accuracy(tess_text, GROUND_TRUTH)
    easy_acc = calculate_accuracy(easy_text, GROUND_TRUTH)
    doctr_acc = calculate_accuracy(doctr_text, GROUND_TRUTH)
    
    print("=" * 75)
    print(f"{'(OCR Engine)':<20} | {'(Confidence)':<24} | {'(Accuracy)':<20}")
    print("=" * 75)
    print(f"{'Tesseract':<20} | {tess_conf * 100:.2f}% {' ':<19} | {tess_acc:.2f}%")
    print(f"{'EasyOCR':<20} | {easy_conf * 100:.2f}% {' ':<19} | {easy_acc:.2f}%")
    print(f"{'docTR':<20} | {doctr_conf * 100:.2f}% {' ':<19} | {doctr_acc:.2f}%")
    print("=" * 75)

C:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


(OCR Engine)         | (Confidence)             | (Accuracy)          
Tesseract            | 81.04%                     | 30.91%
EasyOCR              | 94.10%                     | 92.73%
docTR                | 96.14%                     | 90.91%


In [12]:
print(tess_text)
print(easy_text)
print(doctr_text)
    
    

Test Physical Examination Volume Colour Aspect PH Specific Gravity Deposit Chemical Examination Albumin Sugar Acetone in urine Bilirubin Bile Salts Urobilinogen Other finding RBCs Pus Cells Epithelial Cells Casts Crystals Amorphous Material Bilharzia Ova Other Findings Urine Examination Result Unit Random Sample Amber Yellow os Turbid — 5 1030.0 + Nil Nima 2 Nil Nil Nil _ Normal Trace Mucus (++) Blood (Trace) 6-8 / HPF BO / HPF pte Sol) / HPF Nil
Urine Examination Test Result Unit Physical Examination Volume Random Sample Colour Amber Yellow Aspect Turbid PH Specific Gravity 1030.0 Deposit Chemical Examination Albumin Nil Nil Acetone in urine Nil Bilirubin Nil Bile Salts Nil Urobilinogen Normal Trace Other Mucus Blood (Trace) Microscopic Examination RBCs 6 - 8 HPF Pus Cells 8 - 10 HPF Epithelial Cells + Casts HPF Nil Crystals Nil Amorphous Material Nil Bilharzia Ova Nil Other Nil Sugar finding Findings
Urine Examination Test Result Unit Physical Examination Volume Random Sample Colour 

In [26]:
import os
os.environ["DOCTR_FRAMEWORK"] = "pytorch"

import pytesseract
import easyocr
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from PIL import Image
import Levenshtein

GROUND_TRUTH = "drlogy pathology lab accurate | caring | instant 105 -108, smart vision complex, healthcare road, opposite healthcare complex. mumbai - 689578 0123456789 | 0912345678 drlogypathlab@drlogy.com www.drlogy.com yash m. patel age : 21 years sex : male pid : 555 sample collected at: 125, shivam bungalow, s g road, mumbai ref. by: dr. hiren shah registered on: 02:31 pm 02 dec, 2x collected on: 03:11 pm 02 dec, 2x reported on: 04:35 pm 02 dec, 2x complete blood count (cbc) investigation result reference value unit primary sample type : blood hemoglobin hemoglobin (hb) 12.5 low 13.0 - 17.0 g/dl rbc count total rbc count 5.2 4.5 - 5.5 mill/cumm blood indices packed cell volume (pcv) 57.5 high 40 - 50 % mean corpuscular volume (mcv) calculated 87.75 83 - 101 fl mch calculated 27.2 27 - 32 pg mchc calculated 32.8 32.5 - 34.5 g/dl rdw 13.6 11.6 - 14.0 % wbc count total wbc count 9000 4000-11000 cumm differential wbc count neutrophils 60 50 - 62 % lymphocytes 31 20 - 40 % eosinophils 1 00 - 06 % monocytes 7 00 - 10 % basophils 1 00 - 02 % platelet count platelet count 150000 borderline 150000 - 410000 cumm instruments: fully automated cell counter - mindray 300 interpretation: further confirm for anemia thanks for reference ****end of report**** medical lab technician (dmlt, bmlt) dr. payal shah (md, pathologist) dr. vimal shah (md, pathologist) generated on : 02 dec, 202x 05:00 pm page 1 of 1 sample collection 0123456789"
IMAGE_PATH = r"D:\lenovo\EMAN\1.1.1 Graduation\Graduation-Project\Data Analysis & AI\OCR\tests_analysis\Photos\CBC.jpg"

def clean_and_slice_text(text):
    text = text.lower().strip()
    
    start_index = text.find("unit")
    if start_index != -1:
        text = text[start_index + 4:]
        
    # 2. تحديد نقطة النهاية (عند بداية عبارة end of report)
    # end_index = text.find("end of report")
    # if end_index != -1:
    #     text = text[:end_index]
        
    return text.strip()

def calculate_accuracy(pred_text, ground_truth):
    sliced_pred = clean_and_slice_text(pred_text)
    sliced_ground = clean_and_slice_text(ground_truth)
    
    if not sliced_ground: return 0.0
    
    distance = Levenshtein.distance(sliced_pred, sliced_ground)
    max_len = max(len(sliced_pred), len(sliced_ground))
    return max(0.0, (1 - (distance / max_len)) * 100)

# ----------------- [ 1. Tesseract Engine ] -----------------
def run_tesseract_with_conf(img_path):
    try:
        data = pytesseract.image_to_data(Image.open(img_path), lang='eng', output_type=pytesseract.Output.DICT)
        words = []
        confidences = []
        
        for i in range(len(data['text'])):
            if data['text'][i].strip() != "" and int(data['conf'][i]) != -1:
                words.append(data['text'][i])
                confidences.append(float(data['conf'][i]) / 100.0)
                
        full_text = " ".join(words)
        avg_conf = (sum(confidences) / len(confidences)) if confidences else 0.0
        return full_text, avg_conf
    except Exception as e:
        print(f"Tesseract Error: {e}")
        return "", 0.0

# ----------------- [ 2. EasyOCR Engine ] -----------------
def run_easyocr_with_conf(img_path):
    try:
        reader = easyocr.Reader(['en'], gpu=False)
        result = reader.readtext(img_path, detail=1, paragraph=False)
        
        words = []
        confidences = []
        
        for bbox, text, confidence in result:
            if text.strip() != "":
                words.append(text)
                if confidence is not None:
                    confidences.append(float(confidence))
                
        full_text = " ".join(words)
        avg_conf = (sum(confidences) / len(confidences)) if confidences else 0.0
        return full_text, avg_conf
    except Exception as e:
        print(f"EasyOCR Error: {e}")
        return "", 0.0

# ----------------- [ 3. docTR Engine ] -----------------
def run_doctr_with_conf(img_path):
    try:
        model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
        doc = DocumentFile.from_images(img_path)
        result = model(doc)
        export = result.export()
        
        full_text = []
        all_confidences = [] 
        
        for page in export['pages']:
            for block in page['blocks']:
                for line in block['lines']:
                    words_list = line['words']
                    line_text = " ".join([word['value'] for word in words_list])
                    full_text.append(line_text)
                    for word in words_list:
                        all_confidences.append(word['confidence'])
                        
        combined_text = " ".join(full_text)
        avg_conf = (sum(all_confidences) / len(all_confidences)) if all_confidences else 0.0
        return combined_text, avg_conf
    except Exception as e:
        print(f"docTR Error: {e}")
        return "", 0.0

if __name__ == "__main__":
    
    tess_text, tess_conf = run_tesseract_with_conf(IMAGE_PATH)
    easy_text, easy_conf = run_easyocr_with_conf(IMAGE_PATH)
    doctr_text, doctr_conf = run_doctr_with_conf(IMAGE_PATH)
    
    tess_acc = calculate_accuracy(tess_text, GROUND_TRUTH)
    easy_acc = calculate_accuracy(easy_text, GROUND_TRUTH)
    doctr_acc = calculate_accuracy(doctr_text, GROUND_TRUTH)
    
    print("=" * 75)
    print(f"{'(OCR Engine)':<20} | {'(Confidence)':<24} | {'(Accuracy)':<20}")
    print("=" * 75)
    print(f"{'Tesseract':<20} | {tess_conf * 100:.2f}% {' ':<19} | {tess_acc:.2f}%")
    print(f"{'EasyOCR':<20} | {easy_conf * 100:.2f}% {' ':<19} | {easy_acc:.2f}%")
    print(f"{'docTR':<20} | {doctr_conf * 100:.2f}% {' ':<19} | {doctr_acc:.2f}%")
    print("=" * 75)

C:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


(OCR Engine)         | (Confidence)             | (Accuracy)          
Tesseract            | 86.02%                     | 86.55%
EasyOCR              | 88.14%                     | 84.77%
docTR                | 94.24%                     | 86.51%


In [ ]:
print(tess_text)
print(easy_text)
print(doctr_text)     

DRLOGY PATHOLOGY LAB \. 0123456789 | 0912345678 & Accurate | Caring | Instant © drlogypathlab@drlogy.com 105 -108, SMART VISION COMPLEX, HEALTHCARE ROAD, OPPOSITE HEALTHCARE COMPLEX. MUMBAI - 689578 NS www.drlogy.com Vash M.Patel Sample Gallected At MM 3 4 125, Shi B low, S G Road, ass 02380721 Age ‘21 Years Musabat gee oa Registered on: 02:31 PM 02 Dec, 2X Sex : Male Collected on: 03:11 PM 02 Dec, 2X PID: 555 Ref. By: Dr. Hiren Shah Reported on: 04:35 PM 02 Dec, 2X Complete Blood Count (CBC) Investigation Result Reference Value Unit Primary Sample Type : Blood HEMOGLOBIN Hemoglobin (Hb) 12.5 Low 13.0-17.0 g/dL RBC COUNT Total RBC count 5.2 4.5-5.5 mill/cumm BLOOD INDICES Packed Cell Volume (PCV) 57.5 High 40-50 % Mean Corpuscular Volume (MCV) 87.75 83-101 fL Calculated MCH 27.2 27 - 32 pg Calculated MCHC. 32.8 32.5 - 34.5 g/dL Calculated RDW 13.6 11.6 - 14.0 % WBC COUNT Total WBC count 9000 4000-11000 cumm DIFFERENTIAL WBC COUNT Neutrophils 60 50 - 62 % Lymphocytes 31 20 - 40 % Eosino

In [23]:
import os
os.environ["DOCTR_FRAMEWORK"] = "pytorch"

import pytesseract
import easyocr
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from PIL import Image
import Levenshtein

GROUND_TRUTH = "drlogy pathology lab accurate | caring | instant 105 -108, smart vision complex, healthcare road, opposite healthcare complex. mumbai - 689578 0123456789 | 0912345678 drlogypathlab@drlogy.com www.drlogy.com yashvi m. patel age : 21 years sex : female uhid : 556 sample collected at: 125, shiv complex, s g road, mumbai sample collected by: mr suresh ref. by: dr. hiren shah registered on: 02:31 pm 02 dec, 2x collected on: 03:11 pm 02 dec, 2x reported on: 04:35 pm 02 dec, 2x ferritin investigation result reference value unit sample type serum (3 ml) tat : 2 hrs (normal: 1 - 3 hrs) ferritin clia 365.00 high 22.00 - 322.00 ng/ml note : increase in serum ferritin due to inflammatory conditions (acute phase response) can mask a diagnostically low result comments : serum ferritin appears to be in equilibrium with tissue ferritin and is a good indicator of storage iron in normal subjects and in most disorders. in patients with some hepatocellular diseases, malignancies and inflammatory diseases, serum ferritin is a disproportionately high estimate of storage iron because serum ferritin is an acute phase reactant. in such disorders iron deficiency anemia may exist with a normal serum ferritin concentration. in the presence of inflammation, persons with low serum ferritin are likely to respond to iron therapy. increased levels : • iron overload - hemochromatosis, thalassemia & sideroblastic anemia • malignant conditions - acute myeloblastic & lymphoblastic leukemia, hodgkin's disease & breast carcinoma • inflammatory diseases - pulmonary infections, osteomyelitis, chronic uti, rheumatoid arthritis, sle, burns • acute & chronic hepatocellular disease thanks for reference ****end of report**** medical lab technician (dmlt, bmlt) dr. payal shah (md, pathologist) dr. vimal shah (md, pathologist) to check report authenticity by scanning qr code on top generated on : 02 dec, 202x 05:00 pm page 1 of 1 sample collection 0123456789"
IMAGE_PATH = r"D:\lenovo\EMAN\1.1.1 Graduation\Graduation-Project\Data Analysis & AI\OCR\tests_analysis\Photos\ferritin_high.jpg"

def clean_and_slice_text(text):
    text = text.lower().strip()
    
    start_index = text.find("unit")
    if start_index != -1:
        text = text[start_index + 4:]
        
    # 2. تحديد نقطة النهاية (عند بداية عبارة end of report)
    # end_index = text.find("end of report")
    # if end_index != -1:
    #     text = text[:end_index]
        
    return text.strip()

def calculate_accuracy(pred_text, ground_truth):
    sliced_pred = clean_and_slice_text(pred_text)
    sliced_ground = clean_and_slice_text(ground_truth)
    
    if not sliced_ground: return 0.0
    
    distance = Levenshtein.distance(sliced_pred, sliced_ground)
    max_len = max(len(sliced_pred), len(sliced_ground))
    return max(0.0, (1 - (distance / max_len)) * 100)

# ----------------- [ 1. Tesseract Engine ] -----------------
def run_tesseract_with_conf(img_path):
    try:
        data = pytesseract.image_to_data(Image.open(img_path), lang='eng', output_type=pytesseract.Output.DICT)
        words = []
        confidences = []
        
        for i in range(len(data['text'])):
            if data['text'][i].strip() != "" and int(data['conf'][i]) != -1:
                words.append(data['text'][i])
                confidences.append(float(data['conf'][i]) / 100.0)
                
        full_text = " ".join(words)
        avg_conf = (sum(confidences) / len(confidences)) if confidences else 0.0
        return full_text, avg_conf
    except Exception as e:
        print(f"Tesseract Error: {e}")
        return "", 0.0

# ----------------- [ 2. EasyOCR Engine ] -----------------
def run_easyocr_with_conf(img_path):
    try:
        reader = easyocr.Reader(['en'], gpu=False)
        result = reader.readtext(img_path, detail=1, paragraph=False)
        
        words = []
        confidences = []
        
        for bbox, text, confidence in result:
            if text.strip() != "":
                words.append(text)
                if confidence is not None:
                    confidences.append(float(confidence))
                
        full_text = " ".join(words)
        avg_conf = (sum(confidences) / len(confidences)) if confidences else 0.0
        return full_text, avg_conf
    except Exception as e:
        print(f"EasyOCR Error: {e}")
        return "", 0.0

# ----------------- [ 3. docTR Engine ] -----------------
def run_doctr_with_conf(img_path):
    try:
        model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
        doc = DocumentFile.from_images(img_path)
        result = model(doc)
        export = result.export()
        
        full_text = []
        all_confidences = [] 
        
        for page in export['pages']:
            for block in page['blocks']:
                for line in block['lines']:
                    words_list = line['words']
                    line_text = " ".join([word['value'] for word in words_list])
                    full_text.append(line_text)
                    for word in words_list:
                        all_confidences.append(word['confidence'])
                        
        combined_text = " ".join(full_text)
        avg_conf = (sum(all_confidences) / len(all_confidences)) if all_confidences else 0.0
        return combined_text, avg_conf
    except Exception as e:
        print(f"docTR Error: {e}")
        return "", 0.0

if __name__ == "__main__":
    
    tess_text, tess_conf = run_tesseract_with_conf(IMAGE_PATH)
    easy_text, easy_conf = run_easyocr_with_conf(IMAGE_PATH)
    doctr_text, doctr_conf = run_doctr_with_conf(IMAGE_PATH)
    
    tess_acc = calculate_accuracy(tess_text, GROUND_TRUTH)
    easy_acc = calculate_accuracy(easy_text, GROUND_TRUTH)
    doctr_acc = calculate_accuracy(doctr_text, GROUND_TRUTH)
    
    print("=" * 75)
    print(f"{'(OCR Engine)':<20} | {'(Confidence)':<24} | {'(Accuracy)':<20}")
    print("=" * 75)
    print(f"{'Tesseract':<20} | {tess_conf * 100:.2f}% {' ':<19} | {tess_acc:.2f}%")
    print(f"{'EasyOCR':<20} | {easy_conf * 100:.2f}% {' ':<19} | {easy_acc:.2f}%")
    print(f"{'docTR':<20} | {doctr_conf * 100:.2f}% {' ':<19} | {doctr_acc:.2f}%")
    print("=" * 75)

C:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


(OCR Engine)         | (Confidence)             | (Accuracy)          
Tesseract            | 90.45%                     | 92.80%
EasyOCR              | 86.67%                     | 92.27%
docTR                | 93.50%                     | 93.23%


In [24]:
print(tess_text)
print(easy_text)
print(doctr_text)
    
    

DRLOGY PATHOLOGY LAB X. 0123456789 | 0912345678 y Accurate | Caring | Instant “| drlogypathlab@drlogy.com 105 -108, SMART VISION COMPLEX, HEALTHCARE ROAD, OPPOSITE HEALTHCARE COMPLEX. MUMBAI - 689578 Nt www.drlogy.com Sample Collected At: 125, Shiv complex, S G Road, Mumbai Yashvi M. Patel Age: 21 Years Registered on: 02:31 PM 02 Dec, 2X Sex : Female Sample Collected By: Mr Suresh Collected on: 03:11 PM 02 Dec, 2X UHID : 556 Ref. By: Dr. Hiren Shah Reported on: 04:35 PM 02 Dec, 2X FERRITIN Investigation Result Reference Value Unit Sample Type Serum (3 ml) TAT: 2hrs (Normal: 1 - 3 hrs) a 365.00 High 22.00 - 322.00 ng/mL Note: Increase in serum ferritin due to inflammatory conditions (Acute phase response) can mask a diagnostically low result Comments : Serum ferritin appears to be in equilibrium with tissue ferritin and is a good indicator of storage iron in normal subjects and in most disorders. In patients with some hepatocellular diseases, malignancies and inflammatory diseases, seru